In [1]:
## Load the data
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [2]:
# Generating ground truth
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"
#!wget {PREFIX}/01-agentic-rag/code/rag_helper.py
#!wget {PREFIX}/04-evaluation/code/evaluation_utils.py

In [3]:
## Module Instructions:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [4]:
## For each page, build a JSON user prompt from its filename and content, 
# then call llm_structured with the Questions model. 
# Turn each returned question into a record labeled with the page's filename. 
# The call also returns the token usage, the same as in the lessons.

## Q1: Generating questions for all 72 pages costs money and takes time, so let's start small and generate questions for just the first 3 pages:

- `01-agentic-rag/lessons/01-intro.md`
- `01-agentic-rag/lessons/02-environment.md`
- `01-agentic-rag/lessons/03-rag.md`
Each call returns the token usage, which most LLM APIs report on the response object (e.g. `response.usage.input_tokens` / `prompt_tokens`).

What's the average number of input tokens across these 3 calls?

In [5]:
document_subset = documents[0:3]

In [6]:
## Structure output with pydantic
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]


In [7]:
## Load OpenAI items:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [8]:
## shape the document as json
from evaluation_utils import llm_structured_retry
import json

ground_truth = []
usages = []

for doc in document_subset:  # first 3 pages
    user_prompt = json.dumps(doc)
    result, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )
    for q in result.questions:
        ground_truth.append({"question": q, "document": doc["filename"]})
    usages.append(usage)

In [9]:
usages

[ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=105, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1125),
 ResponseUsage(input_tokens=1286, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=117, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1403),
 ResponseUsage(input_tokens=1753, input_tokens_details=InputTokensDetails(cached_tokens=1280, cache_write_tokens=0), output_tokens=95, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1848)]

In [10]:
avg_input_tokens = sum(u.input_tokens for u in usages) / len(usages)
print(f"Average input tokens: {avg_input_tokens}")

Average input tokens: 1353.0


## Q1 answer: 1400

### Full Ground Truth

In [11]:
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"
#!wget {PREFIX}/cohorts/2026/04-evaluation/ground-truth.csv

# Load it with pandas into a list of records called ground_truth. 
# Each record has a question and the filename of the page that should answer it.


In [12]:
import pandas as pd
ground_truth = pd.read_csv('ground-truth.csv')

In [13]:
ground_truth.head(2)

,question,filename
0,What exactly is a retrieval-augmented generati...,01-agentic-rag/lessons/01-intro.md
1,Why does this course build the RAG project in ...,01-agentic-rag/lessons/01-intro.md


In [14]:
# Searching through chucks
# Create the chunks:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [15]:
# Now rebuild the search from homework 2 over these chunks. 
# Build a text index (Index) and a vector index (VectorSearch), both keyed on filename. 
# Wrap each one in a function, text_search and vector_search, that takes a query and the number of results to return (5 by default).

In [16]:
from minsearch import Index
def text_search(documents, text_fields, keyword_fields, num_results=5):
    index = Index(text_fields=text_fields, keyword_fields=keyword_fields)
    index.fit(documents)

    def search(query, num_results=num_results):
        return index.search(query, num_results=num_results)

    return search

In [17]:
text_search = text_search(chunks, text_fields=["content"], keyword_fields=["filename"])

## Q2: After running text_search for it, what's the filename of the first result?

- 01-agentic-rag/lessons/01-intro.md
- 01-agentic-rag/lessons/03-rag.md
- 01-agentic-rag/lessons/13-function-calling.md
- 01-agentic-rag/lessons/10-rag-next-steps.md

In [18]:
text_search(ground_truth.iloc[0]["question"])

[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

## Q2 Answer: 
`01-agentic-rag/lessons/03-rag.md`

In [19]:
import sys
sys.path.insert(0, '/workspaces/llm-zoomcamp-rag/llm-zoomcamp-onnx')

In [20]:
from embedder import Embedder
embed = Embedder('/workspaces/llm-zoomcamp-rag/llm-zoomcamp-onnx/models/Xenova/all-MiniLM-L6-v2')

2026-07-12 15:47:08.215638595 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [21]:
import numpy as np
from minsearch import VectorSearch

def vector_search(documents, embedder, keyword_fields, num_results=5):
    vectors = np.array([embedder.encode(doc['content']) for doc in documents])
    index = VectorSearch(keyword_fields=keyword_fields)
    index.fit(vectors, documents)

    def search(query, num_results=num_results):
        query_vector = embedder.encode(query)
        return index.search(query_vector, num_results=num_results)

    return search

In [22]:
vector_search = vector_search(chunks, embed, keyword_fields=["filename"])

In [23]:
#For hybrid search, reuse the rrf function from homework 2:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [24]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

## Q3: After running `vector_search` for the same question, what's the filename of the first result?

- `01-agentic-rag/lessons/01-intro.md`
- `01-agentic-rag/lessons/03-rag.md`
- `04-evaluation/lessons/11-evaluation-intro.md`
- `04-evaluation/lessons/12-rag-answers.md`

In [25]:
# Q3. First result with vector search:
vector_search(ground_truth.iloc[0]["question"])

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

## Q3 Answer: `01-agentic-rag/lessons/01-intro.md`